# 🧩 PySweeper – OOP-Kurs

Du baust `main.py` in ein objektorientiertes Paket um, **eine Klasse pro Datei**, so wie es `pysweeper-class-diagram.mmd` vorgibt.
Den Code schreibst **du**, in echten `.py`-Dateien in PyCharm. Dieses Notebook ist dein Kursbuch und dein Prüfer.

**So läuft jede Lektion ab** (wie bei JetBrains Academy):

| | |
|---|---|
| 🎯 **Lernziel** | was du danach kannst |
| 📖 **Theorie** | Python-Konzept mit Java-Vergleich und Beispiel zum Ausführen (mit anderem Thema, damit nichts verraten wird) |
| 🛠 **Aufgabe** | der Vertrag: welche Klasse, welche Attribute, welches Verhalten |
| 💡 **Hinweise** | `hint("l3", 1)` → Stufe 1 bis 3, vom Denkanstoß bis zum Gerüst. Die entscheidenden Zeilen schreibst du immer selbst. |
| ✅ **Check** | `check("l3")` prüft deinen aktuellen Code. Rot ist normal, rot zeigt den nächsten Schritt. |
| 🧪 **Spielwiese** | ausprobieren, was du gebaut hast |
| 🤔 **Warum?** | kurze Fragen zum Selbsterklären, danach `explain("l3")` zum Vergleichen |

**Spielregeln**
- `main.py` bleibt unverändert, sie ist deine Vorlage (dein „Worked Example“). Du **portierst** die Logik, statt sie neu zu erfinden.
- Das Diagramm ist der Vertrag. Koordinaten sind immer `(x, y)` = (Spalte, Zeile). Das ist **genau andersherum** als `(r, c)` in `main.py`.
- Erst selbst versuchen, dann Hinweise. Vor jeder neuen Hinweis-Stufe kurz formulieren, wo du hängst.
- Nach jedem grünen Check: committen (der Check schlägt dir die Nachricht vor).

**Kernpfad ≈ 4½ h** (L0–L6 ohne ➕). Die ➕-Erweiterungen und Lektion 7 sind für „falls Zeit“, gerne auch an einem zweiten Tag.

| Lektion | Thema | ⏱ Kern | ➕ falls Zeit | |
|---|---|---|---|---|
| 0 | Diagramm lesen, Paketstruktur | 15' | | `check("l0")` |
| 1 | `GameState`, `DifficultyConfig`: Enum & dataclass | 30' | | `check("l1")` |
| 2 | `Cell`: `__init__`, `self`, Instanzattribute | 25' | | `check("l2")` |
| 3 | `Board` I: Komposition, Generator, Zufall | 45' | | `check("l3")` |
| 4 | `Board` II: Verhalten, Cascade, Abfragen | 45' | Chording 20' | `check("l4")`, `check("l4_extra")` |
| 5 | `Game`: Controller & Zustandsautomat | 35' | Timer & Zähler 20' | `check("l5")`, `check("l5_extra")` |
| 6 | `UserInterface` (ABC) & `TerminalUI` | 75' | M-Menü | `check("l6")` |
| 7 | Bonus: `TkinterUI`, Diagramm-Reflexion, Abschluss-Quiz | | 45' | `check("l7")` |
| 8 | `StartMenu`: von `tk.Frame` erben, `super()`, Funktionen im `dict` | 45' | Zurück ins Menü 15' | `check("l8")`, `check("l8_extra")` |

Ohne Chording fehlt dem Terminal-Spiel in L6 nur dieses Komfort-Feature. Den Timer kannst du dort als `0` anzeigen, bis L5➕ erledigt ist.

### ⚙️ Setup
Kernel/Interpreter: **`.venv` (Python 3.12.10)** des Projekts, dieselbe Version wie auf der Arbeit. Führe die nächste Zelle einmal pro Sitzung aus.
Verhält sich die Spielwiese seltsam (alte Klassen-Versionen), starte den Kernel neu. Die Checks laufen immer in einem frischen Prozess und sehen deshalb stets deinen aktuellen Code.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
for path in (ROOT, ROOT / "course"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from _check import check, hint, explain
print(f"Projekt: {ROOT}\nPython:  {sys.version.split()[0]}  ✔ bereit")

---
## Lektion 0 – Das Diagramm lesen & die Paketstruktur anlegen

🎯 **Lernziel:** Du liest ein UML-Klassendiagramm in Mermaid-Notation und weißt, wie Python Module und Pakete organisiert.

### 📖 Mermaid-Klassendiagramm in 60 Sekunden

| Notation | Bedeutung | Im PySweeper-Diagramm |
|---|---|---|
| `A *-- B` | **Komposition**: A besitzt B, B lebt nicht ohne A | `Game *-- Board`, `Board *-- Cell` |
| `A --> B` | **Assoziation**: A kennt B dauerhaft | `Game --> DifficultyConfig` |
| `A <\|-- B` | **Vererbung**: B ist ein A | `UserInterface <\|-- TerminalUI` |
| `A ..> B` | **Abhängigkeit**: A benutzt B kurz (z. B. als Parameter) | `UserInterface ..> Game` |
| `+name` / `-name` | öffentlich / privat | `-_plant_mines()` |
| `methode()*` | abstrakte Methode | `+run(game)*` |
| `<<enumeration>>`, `<<abstract>>` | Stereotyp | `GameState`, `UserInterface` |

```mermaid
classDiagram
    direction LR
    Game *-- Board : manages layout
    Game --> DifficultyConfig : configured by
    Game --> GameState : tracks
    Board *-- Cell : composed of 2D grid
    UserInterface <|-- TerminalUI
    UserInterface <|-- TkinterUI
    UserInterface ..> Game : calls reveal / toggle_flag
    UserInterface ..> Board : reads data from
```
*(Das vollständige Diagramm mit allen Attributen: `pysweeper-class-diagram.mmd`. PyCharm zeigt es mit dem Mermaid-Plugin an.)*

### 📖 Module & Pakete: Python vs. Java

| Java | Python |
|---|---|
| Datei `Cell.java`, **genau eine** public class | Datei `../pysweeper/cell.py` (snake_case) = **Modul**, beliebig viele Klassen erlaubt |
| Ordner = package, `package com.example.pysweeper;` | Ordner mit `../pysweeper/__init__.py` = **Paket**, kein package-Statement |
| `import com.example.pysweeper.Cell;` | `from pysweeper.cell import Cell` |
| `public static void main` | `python -m pysweeper` führt `pysweeper/__main__.py` aus (Lektion 6) |

„Eine Klasse pro Datei“ ist in Python **keine** Pflicht. Wir machen es hier als Übung: An den Imports jeder Datei siehst du dann sofort, wer wen kennt.

### 🛠 Aufgabe 0
Lege neben `main.py` diese Struktur an, mit **leeren** Dateien:
```
pysweeper/
├── __init__.py
├── game_state.py           # GameState
├── difficulty_config.py    # DifficultyConfig + EASY / MEDIUM / HARD / DIFFICULTIES
├── cell.py                 # Cell
├── board.py                # Board
├── game.py                 # Game
└── ui/
    ├── __init__.py
    ├── user_interface.py   # UserInterface
    └── terminal_ui.py      # TerminalUI
```

**✍️ Abruf-Übung** (bevor du weiterliest, in die Zelle darunter schreiben):
1. Wie würdest du „`Game` hält dauerhaft eine `TerminalUI`“ notieren, und wie, wenn die UI dem Game *gehören* würde?
2. Warum zeigt der Pfeil `UserInterface ..> Game` von der UI zum Spiel und nicht umgekehrt?
3. Welche Methoden von `Board` sind privat, und woran erkennst du das im Diagramm?

*Deine Antworten:*
1. `UserInterface(TerminalUI).run(Game(EASY))`. Wenn UI dem Game gehören würde: `Game(difficulty_condif="EASY", user_interace="TerminalUI")`.
2. Weil die Klasse `UserInterface` entweder ein GUI (`TkinterUI`) oder TUI (`TerminalUI`) an `Game` übermittelt. `Game` selbst kennt die UI-Optionen (`TkinterUI, TerminalUI`) nicht.
3. In einem Python Klassendiagramm gibt es zwei möglichkeiten eine private Methode(/Attribute) zu erkennen. Einmal die Python Naming Convention welche besagt, dass private Methoden, Attribute & Variablen mit einem "_var_name" beginnen sollten (nicht von Interpreter kontrolliert). Aber für ein Klassendiagramm gilt, eine private Methode(/Attribute) wird mit dem Prefix "-" aufgelistet.

In [ ]:
check("l0")

In [ ]:
# hint("l0", 1)

In [ ]:
# explain("l0")   # erst nach deinen Antworten

---
## Lektion 1 – `GameState` & `DifficultyConfig`

🎯 **Lernziel:** Du ersetzt „Magic Strings“ und Dicts durch `Enum` und `@dataclass(frozen=True)` und weißt, was Python dir dabei generiert.

### 📖 Theorie

| Java | Python |
|---|---|
| `enum TrafficLight { RED, YELLOW, GREEN }` | `class TrafficLight(Enum): RED = auto() …` |
| `light == TrafficLight.RED` | `light is TrafficLight.RED` (Mitglieder sind Singletons) |
| `record Ticket(String movie, int seat) {}` | `@dataclass(frozen=True) class Ticket: movie: str; seat: int` |
| Typen werden vom Compiler erzwungen | Type Hints sind **Dokumentation** (+ IDE/mypy), zur Laufzeit nicht geprüft |

`main.py` nutzt `'playing'`/`'won'`/`'lost'` und verschachtelte Dicts (`difficulty['settings']['board']`). Beides ist fehleranfällig: Ein Tippfehler fällt erst zur Laufzeit auf, wenn überhaupt.

In [ ]:
from enum import Enum, auto
from dataclasses import dataclass

class TrafficLight(Enum):
    RED = auto()          # auto() vergibt die Werte, uns interessieren nur die Namen
    YELLOW = auto()
    GREEN = auto()

light = TrafficLight.RED
print(light, "|", light.name, "|", light is TrafficLight.RED)
print([l.name for l in TrafficLight])      # über alle Mitglieder iterieren


@dataclass(frozen=True)
class Ticket:
    movie: str
    seat: int
    price: float = 9.5    # Default-Wert

t = Ticket("Dune", 12)
print(t)                                   # __repr__ geschenkt
print(t == Ticket("Dune", 12))             # __eq__ vergleicht Werte → True
try:
    t.seat = 13
except Exception as e:
    print(type(e).__name__, "→", e)

### 🛠 Aufgabe 1

**`pysweeper/game_state.py`**: `GameState` ist ein `Enum` mit genau `PLAYING`, `WON`, `LOST`.

**`pysweeper/difficulty_config.py`**:

| Element | Vertrag |
|---|---|
| `DifficultyConfig` | `@dataclass(frozen=True)`, Felder **in dieser Reihenfolge** `name: str`, `amount: int`, `size: tuple[int, int]` |
| `size` | `(width, height)`, siehe Notiz im Diagramm |
| `EASY` / `MEDIUM` / `HARD` | Namen `"Easy"`, `"Medium"`, `"Hard"`; Werte aus `main.py`: 9×9/10, 16×16/40, **30 breit × 16 hoch**/99 |
| `DIFFICULTIES` | Liste `[EASY, MEDIUM, HARD]` |

In [ ]:
check("l1")

In [ ]:
# hint("l1", 1)

### 🧪 Spielwiese

In [ ]:
from pysweeper.game_state import GameState
from pysweeper.difficulty_config import DIFFICULTIES, HARD

print(list(GameState))
for d in DIFFICULTIES:
    width, height = d.size          # Tuple-Unpacking
    print(f"{d.name:<7} {width}x{height}  {d.amount} Minen")

### 🤔 Warum?
1. Warum ist ein `Enum` sicherer als die Strings `'won'`/`'lost'` aus `main.py`?
2. Was könnte schiefgehen, wenn `EASY` **nicht** frozen wäre und zwei Spiele gleichzeitig `EASY` benutzen?

In [ ]:
# explain("l1")

---
## Lektion 2 – `Cell`

🎯 **Lernziel:** Du schreibst eine Klasse mit `__init__` und `self`, unterscheidest Instanz- von Klassenattributen und kapselst eine Regel in einer Methode.

### 📖 Theorie

| Java | Python |
|---|---|
| Felder werden deklariert: `private boolean booked;` | Attribute **entstehen** durch Zuweisung in `__init__`: `self.is_booked = False` |
| Konstruktor `Seat(int row) { this.row = row; }` | `def __init__(self, row): self.row = row` |
| `this` ist implizit | `self` ist **expliziter erster Parameter** jeder Methode |
| `static int count;` | Attribut direkt im Klassenrumpf = **Klassenattribut** |
| `toString()` | `__repr__()` |
| Überladene Konstruktoren | gibt es nicht, stattdessen Default-Argumente |

In [ ]:
class Seat:
    hall = "Saal 1"                  # Klassenattribut, gehört der Klasse (≈ static)

    def __init__(self, row: int, number: int):
        self.row = row               # Instanzattribute entstehen durch Zuweisung an self
        self.number = number
        self.is_booked = False

    def book(self) -> None:
        if self.is_booked:           # Regel gekapselt: doppelt buchen geht nicht
            print("schon gebucht")
            return
        self.is_booked = True

    def __repr__(self) -> str:       # ≈ toString()
        return f"Seat(row={self.row}, number={self.number}, booked={self.is_booked})"

a, b = Seat(1, 1), Seat(1, 2)
a.book(); a.book()
print(a, b, sep="\n")
print(vars(a))                       # nur die Instanzattribute
print(Seat.hall, a.hall)             # Klassenattribut über Klasse oder Objekt lesbar

⚠️ **Falle:** Ein *veränderbares* Klassenattribut teilen sich alle Objekte:

In [ ]:
class Playlist:
    songs = []                       # EINE Liste für ALLE Playlists

    def add(self, song):
        self.songs.append(song)

p1, p2 = Playlist(), Playlist()
p1.add("Song A")
print(p2.songs)                      # ['Song A'], obwohl wir nur p1 geändert haben

### 🛠 Aufgabe 2: `pysweeper/cell.py`
Diesmal als Lückentext. Fülle das Gerüst in deiner Datei aus:
```python
class Cell:
    def __init__(self, ___, ___):
        self.x = ___
        self.y = ___
        self.is_mine = ___
        self.is_revealed = ___
        self.is_flagged = ___
        self.neighbor_mines = ___

    def toggle_flag(self) -> None:
        ___   # aufgedeckte Zellen lassen sich nicht markieren (siehe main.py: toggle_flag)
```
Optional: ein `__repr__`, das macht das Debuggen in den nächsten Lektionen angenehmer.

In [ ]:
check("l2")

In [ ]:
# hint("l2", 1)

### 🧪 Spielwiese

In [ ]:
from pysweeper.cell import Cell

c = Cell(3, 5)
c.toggle_flag()
print(c, vars(c))

### 🤔 Warum?
1. Warum steht `self` explizit in jeder Methodensignatur, was macht Python bei `cell.toggle_flag()` intern?
2. Warum gehört die Regel „aufgedeckt → nicht markierbar“ in `Cell` und nicht ins `Board`?

In [ ]:
# explain("l2")

---
## Lektion 3 – `Board` I: Aufbau

🎯 **Lernziel:** Du setzt Komposition um (Board erzeugt und besitzt seine Zellen), schreibst einen Generator mit `yield` und kennst die Python-Konvention für „privat“.

### 📖 Theorie

| Java | Python |
|---|---|
| `private void plantMines()` | `def _plant_mines(self)`: Unterstrich = „intern“, **nur Konvention** (`__name` würde zusätzlich Name-Mangling auslösen, ist hier aber nicht nötig) |
| `Iterator<int[]>` / `Stream` | **Generator**: Funktion mit `yield` liefert Werte einzeln bei Bedarf |
| `Collections.shuffle(list); list.subList(0, k)` | `random.sample(liste, k)` |
| `new Cell[h][w]` + Schleifen | verschachtelte **List Comprehension** |

In [ ]:
# ⚠️ Falle: * kopiert Referenzen, nicht Objekte
grid = [[0] * 3] * 2
grid[0][0] = 99
print(grid, "| gleiche Zeile?", grid[0] is grid[1])

# ✔ Jede Zeile einzeln erzeugen
grid = [[0 for x in range(3)] for y in range(2)]
grid[0][0] = 99
print(grid)

In [ ]:
def adjacent_seats(row, number, rows=3, numbers=4):
    # Liefert die Plätze links/rechts/vorne/hinten, die es im Saal wirklich gibt.
    for dr, dn in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        r, n = row + dr, number + dn
        if 0 <= r < rows and 0 <= n < numbers:
            yield r, n

print(list(adjacent_seats(0, 0)))       # Ecke: nur 2 Nachbarn
gen = adjacent_seats(1, 1)
print(next(gen), next(gen))             # Werte werden erst bei Bedarf berechnet
print({(0, 0), *adjacent_seats(0, 0)})  # Generator direkt in ein Set entpacken

In [ ]:
import random

class Cinema:
    def __init__(self, rows, numbers):
        self.rows, self.numbers = rows, numbers
        self.seats = [[Seat(r, n) for n in range(numbers)] for r in range(rows)]  # Komposition

    def _free_seats(self):                # intern
        return [s for row in self.seats for s in row if not s.is_booked]

    def book_random(self, k):
        for seat in random.sample(self._free_seats(), k):   # k verschiedene, ohne Zurücklegen
            seat.book()

cinema = Cinema(3, 4)                     # braucht Seat aus Lektion 2
cinema.book_random(5)
print(sum(s.is_booked for row in cinema.seats for s in row), "gebucht")  # True zählt als 1

### 🛠 Aufgabe 3: `pysweeper/board.py` (Aufbau)

| Element | Vertrag |
|---|---|
| `Board(width, height, mine_amount)` | speichert die drei Werte, `mines_planted = False`, ruft `_initialize_board()` auf |
| `_initialize_board()` | baut `grid`: Liste aus `height` Zeilen à `width` **eigenen** `Cell`-Objekten → `grid[y][x]` |
| `_plant_mines(safe_x, safe_y)` | legt genau `mine_amount` Minen **zufällig**; Startzelle + Nachbarn bleiben frei (bei zu wenig Platz nur die Startzelle); setzt `mines_planted = True` |
| `_calculate_neighbors()` | `neighbor_mines` jeder Nicht-Mine = Minen in den **8** Nachbarfeldern; **setzt** den Wert (mehrfach aufrufbar) |
| *(empfohlen, nicht im Diagramm)* | privater Generator `_neighbors(x, y)` |

**Übersetzungshilfe `main.py` → `Board`**

| `main.py` | neu |
|---|---|
| `(r, c)` | `(x, y)` = `(c, r)` ⚠️ |
| `self.board[r, c] == '*'` | `self.grid[y][x].is_mine` |
| `self.board[r, c] = str(count)` | `cell.neighbor_mines = count` |
| `neighbours(r, c)` | `_neighbors(x, y)` |
| `np.random.default_rng().choice(...)` | `random.sample(...)` (kein numpy mehr nötig) |

In [ ]:
check("l3")

In [ ]:
# hint("l3", 1)

### 🧪 Spielwiese

In [ ]:
from pysweeper.board import Board

b = Board(9, 9, 10)
b._plant_mines(4, 4)
b._calculate_neighbors()
for row in b.grid:
    print(" ".join("*" if c.is_mine else str(c.neighbor_mines) for c in row))

### 🤔 Warum?
1. Warum funktioniert `[[Cell(0, 0)] * width] * height` nicht, obwohl es auf den ersten Blick ein 2D-Gitter ergibt?
2. Warum werden die Minen erst beim ersten Klick gelegt und nicht schon in `__init__`?
3. Was gewinnst du durch **einen** Generator `_neighbors` gegenüber drei Kopien der Doppelschleife?

In [ ]:
# explain("l3")

---
## Lektion 4 – `Board` II: Verhalten

🎯 **Lernziel:** Du implementierst Verhalten mit klarer Trennung von *Command* und *Query*, schreibst einen iterativen Flood-Fill und kennst Pythons Rekursionsgrenze.

### 📖 Theorie
**Command-Query-Separation:** Eine Methode *ändert* Zustand (`reveal_cell`, `toggle_flag`) **oder** *beantwortet eine Frage* (`check_win`), nicht beides. Abfragen kann man dann gefahrlos beliebig oft aufrufen.

**Rekursion:** Python hat (wie Java) keine Tail-Call-Optimierung, aber ein viel kleineres Limit:

In [ ]:
import sys
print("Rekursionslimit:", sys.getrecursionlimit())

def depth(n):
    return 0 if n == 0 else 1 + depth(n - 1)

try:
    depth(5000)
except RecursionError as e:
    print("RecursionError:", e)

Die Lösung ist ein **expliziter Stack** (eine Liste mit `append`/`pop`, ≈ `Deque` in Java). Beispiel mit anderem Thema: Wer ist über Freundschaften erreichbar?

In [ ]:
friends = {
    "Anna": ["Ben", "Cem"], "Ben": ["Anna", "Dora"], "Cem": ["Anna"],
    "Dora": ["Ben"], "Emil": ["Fynn"], "Fynn": ["Emil"],
}

def reachable(start):
    seen = set()
    stack = [start]
    while stack:
        person = stack.pop()
        if person in seen:          # schon besucht → überspringen (sonst Endlosschleife!)
            continue
        seen.add(person)
        stack.extend(friends[person])
    return seen

print(reachable("Anna"))           # Emil und Fynn sind nicht erreichbar

### 🛠 Aufgabe 4: `Board` (Verhalten)
Dein eigener Entwurf ist der Vertrag:

| Methode | Vertrag |
|---|---|
| `reveal_cell(x, y) -> bool` | **`True` = Zelle wurde aufgedeckt.** Markiert → `False` (und keine Minen legen!). Schon offen → `False`. Sonst: falls `not mines_planted` → `_plant_mines(x, y)` + `_calculate_neighbors()`; dann `_cascade_reveal(x, y)`; `True`. Eine Mine wird ebenfalls aufgedeckt (`True`); ob das Spiel verloren ist, entscheidet später `Game`. |
| `_cascade_reveal(x, y)` | Flood-Fill **mit Stack**: deckt `(x, y)` **selbst** auf; nur bei einer **Nicht-Mine** mit `neighbor_mines == 0` geht es bei den Nachbarn weiter (über Nullen hinweg, Stopp an Zahlen). Offene und markierte Zellen werden übersprungen. |
| `toggle_flag(x, y)` | delegiert an `Cell.toggle_flag()` |
| `check_win() -> bool` | `True` genau dann, wenn alle Nicht-Minen offen sind. **Reine Abfrage.** |

In [ ]:
check("l4")

In [ ]:
# hint("l4", 1)

### ➕ Erweiterung: Chording & Abfragen
`main.py` kann mehr als das Diagramm: Klickt man auf eine offene Zahl, deren Flaggen alle gesetzt sind, gehen die restlichen Nachbarn auf („Chording“).
Dabei kann eine **falsch gesetzte Flagge** eine Mine an einer *anderen* Stelle als der angeklickten aufdecken. Deshalb brauchen wir eine Abfrage, die das ganze Brett prüft.

📖 `@property` ≈ Getter, aber Zugriff ohne Klammern. `sum()`/`any()` über Generatoren zählen bzw. prüfen kompakt:

In [ ]:
class Thermostat:
    def __init__(self, readings):
        self.readings = readings

    @property
    def too_hot(self) -> int:                # berechnet, aber Zugriff wie ein Attribut
        return sum(r > 25 for r in self.readings)

    def any_frost(self) -> bool:             # Abfrage-Methode
        return any(r < 0 for r in self.readings)

t = Thermostat([21, 27, -2, 30])
print(t.too_hot, t.any_frost())              # too_hot ohne Klammern!

| Element | Vertrag |
|---|---|
| `flags_placed` | `@property`, Anzahl markierter Zellen |
| `mine_revealed() -> bool` | `True`, sobald irgendeine Mine aufgedeckt ist |
| Chording in `reveal_cell` | auf offener Zahl, wenn #Flaggen ringsum == `neighbor_mines`: alle verdeckten, unmarkierten Nachbarn aufdecken (inkl. Cascade) → `True`, wenn etwas aufging, sonst `False` |

In [ ]:
check("l4_extra")

In [ ]:
# hint("l4_extra", 1)

### 🤔 Warum?
1. Warum ist der Stack hier nicht nur Geschmackssache? (Tipp: Wie tief kann die Rekursion werden, und wie groß darf ein `Board` sein?)
2. Warum soll `check_win` nichts verändern, obwohl `main.py` bei Sieg alle Minen automatisch markiert?
3. Warum gibt `reveal_cell` bei einer Flagge `False` zurück, statt eine Exception zu werfen?

In [ ]:
# explain("l4")

---
## Lektion 5 – `Game`: der Controller

🎯 **Lernziel:** Du baust einen Controller, der einen Zustandsautomaten verwaltet und Arbeit an ein komponiertes Objekt delegiert.

### 📖 Theorie
Ein **Zustandsautomat** erlaubt Aktionen nur in bestimmten Zuständen. In Python geht das elegant mit *Guard Clauses* (früher `return`):

In [ ]:
from enum import Enum, auto

class OrderStatus(Enum):
    OPEN = auto()
    PAID = auto()

class Order:
    def __init__(self, items):
        self.items = list(items)
        self.status = OrderStatus.OPEN

    def add(self, item):
        if self.status is not OrderStatus.OPEN:   # Guard Clause
            return
        self.items.append(item)

    def pay(self):
        if self.status is not OrderStatus.OPEN:
            return
        self.status = OrderStatus.PAID

o = Order(["Popcorn"])
o.pay()
o.add("Cola")                                     # ignoriert, schon bezahlt
print(o.status, o.items)

**Delegation:** `Game` deckt nicht selbst Zellen auf, sondern sagt `self.board.reveal_cell(x, y)` und entscheidet anhand der Antwort über den Zustand. Aufgabenteilung wie zwischen Controller und Service in Spring.

### 🛠 Aufgabe 5: `pysweeper/game.py`

| Element | Vertrag |
|---|---|
| `Game(difficulty)` | speichert `difficulty`, `state = GameState.PLAYING`, baut `board = Board(width, height, amount)` aus `difficulty.size`/`.amount` |
| `reveal(x, y)` | nur bei `PLAYING`. Dein Entwurf: wenn `board.reveal_cell` `True` liefert → Mine? `LOST` : `check_win()`? `WON` |
| `toggle_flag(x, y)` | nur bei `PLAYING`, delegiert ans Board |
| `reset_board()` | neues Board mit gleicher Schwierigkeit, `state = PLAYING` |
| `set_difficulty(new)` | neue Schwierigkeit übernehmen + zurücksetzen |

⚠️ Die Tests ersetzen `game.board` teilweise durch ein vorbereitetes Board mit festen Minen. `Game` soll deshalb bei jedem Zug frisch über `self.board` arbeiten und nichts vom Board zwischenspeichern.

In [ ]:
check("l5")

In [ ]:
# hint("l5", 1)

### ➕ Erweiterung: Timer, Minen-Zähler, Chording-Verlust
*Setzt L4➕ voraus (`flags_placed`, `mine_revealed()`).*

📖 Für Stoppuhren `time.monotonic()` (≈ `System.nanoTime()`) statt `time.time()`. Die Wanduhr kann springen (manuelles Stellen der Uhr, NTP-Sync), die monotone Uhr läuft nur vorwärts.

In [ ]:
import time
start = time.monotonic()
time.sleep(0.3)
print(f"{time.monotonic() - start:.2f} s vergangen")

| Element | Vertrag |
|---|---|
| `mines_left` | `@property`: `mine_amount - board.flags_placed` |
| `elapsed` | `@property`, ganze Sekunden, **abgerundet** (`int(...)`): `0` vor dem ersten erfolgreichen `reveal`; läuft danach; bleibt bei `WON`/`LOST` stehen. Startzeit beim ersten erfolgreichen `reveal` merken (`start_time`, anfangs `None`), **nicht** an `board.mines_planted` koppeln; `time.monotonic()` über `import time` aufrufen |
| `reset_board()` | setzt auch den Timer zurück |
| Verlust | über `board.mine_revealed()`, so wird auch Chording auf eine Mine erkannt |

In [ ]:
check("l5_extra")

In [ ]:
# hint("l5_extra", 1)

### 🧪 Spielwiese: Minesweeper im Notebook, ganz ohne UI

In [ ]:
from pysweeper.game import Game
from pysweeper.difficulty_config import EASY

def show(game):
    # Textansicht eines Games, nutzt nur öffentliche Attribute aus dem Diagramm.
    for row in game.board.grid:
        line = []
        for cell in row:
            if cell.is_flagged:
                line.append("F")
            elif not cell.is_revealed:
                line.append(".")
            elif cell.is_mine:
                line.append("*")
            else:
                line.append(str(cell.neighbor_mines or " "))
        print(" ".join(line))
    print(game.state)

game = Game(EASY)
game.reveal(4, 4)
show(game)

In [ ]:
# Weiterspielen: game.reveal(x, y) / game.toggle_flag(x, y), dann show(game)

### 🤔 Warum?
1. Warum entscheidet `Game` über `LOST`/`WON` und nicht das `Board`?
2. In den Tests wird `game.board` einfach durch ein vorbereitetes Board ersetzt. Warum funktioniert das, und ist das schon Dependency Injection wie in Spring?

In [ ]:
# explain("l5")

---
## Lektion 6 – `UserInterface` (ABC) & `TerminalUI`

🎯 **Lernziel:** Du definierst eine abstrakte Basisklasse, nutzt Polymorphie und hältst die Abhängigkeitsrichtung UI → Logik ein.

### 📖 Theorie

| Java | Python |
|---|---|
| `abstract class Notifier { abstract void send(String m); }` | `class Notifier(ABC):` + `@abstractmethod def send(self, m): ...` |
| `interface` | ABC oder `typing.Protocol` (rein strukturell, Duck Typing mit Typprüfung) |
| `new Notifier()` → Compilerfehler | `Notifier()` → `TypeError` zur Laufzeit |
| `public static void main(String[] a)` | `if __name__ == "__main__":` |

In [ ]:
from abc import ABC, abstractmethod

class Notifier(ABC):
    @abstractmethod
    def send(self, message: str) -> None: ...

    def send_all(self, messages):            # konkrete Methode nutzt die abstrakte
        for m in messages:
            self.send(m)

class ConsoleNotifier(Notifier):
    def send(self, message):
        print("📣", message)

class ShoutNotifier(Notifier):
    def send(self, message):
        print("📣", message.upper() + "!!!")

for notifier in (ConsoleNotifier(), ShoutNotifier()):     # Polymorphie
    notifier.send_all(["Film startet", "Popcorn ist fertig"])

try:
    Notifier()
except TypeError as e:
    print("TypeError:", e)

**Abhängigkeitsrichtung:** `../pysweeper/ui` darf `../pysweeper/game.py` importieren, **nie umgekehrt**. Dann bleibt die Logik ohne Terminal testbar, und eine zweite UI (Lektion 7) braucht keine Änderung an `Game`.

```
TerminalUI ──┐
             ├──▶ Game ──▶ Board ──▶ Cell
TkinterUI ───┘
```

### 🛠 Aufgabe 6

| Element | Vertrag |
|---|---|
| `../pysweeper/ui` | `UserInterface(ABC)` mit abstrakten `run(game)` und `draw_board(board)` |
| `TerminalUI(stdscr)` | erbt von `UserInterface`; `__init__` **speichert nur**: `stdscr`, `cursor_x = cursor_y = 0` (`run` darf den Cursor danach zentrieren) |
| `run(game)` | merkt sich `self.game`, ruft `_setup_curses()` auf, Spielschleife (Port von `main.py: play`) |
| `_move_cursor(dx, dy)` | begrenzt den Cursor auf `self.game.board` (x ↔ width, y ↔ height) |
| `draw_board(board)` | Port von `main.py: draw` / `cell_look`: Zellen aus `board.grid`, Titel/Status/Meldungen aus `self.game` (`difficulty.name`, `mines_left`, `elapsed`, `state`) |
| `_get_input()` | liefert die gedrückte Taste (`stdscr.getch()`) |
| `pysweeper/__main__.py` | startet per `curses.wrapper(...)` eine `TerminalUI` mit `Game(EASY)`, **nur** unter `if __name__ == "__main__":` |

Tasten wie in `main.py`: Pfeile/WASD, Leertaste/Enter, F, R (→ `game.reset_board()`), Q. *Optional:* M-Menü → `game.set_difficulty(...)` (danach Cursor neu setzen und `stdscr.timeout(250)` wiederherstellen).
Tipp zu Type Hints: In Python 3.12 werden Annotationen **sofort** beim `def` ausgewertet. `def run(self, game: Game)` braucht also ein `from pysweeper.game import Game`, sonst gibt es einen `NameError`. Für Klassen, die man nur für die Annotation importieren würde, gibt es zwei Wege: `from __future__ import annotations` als erste Zeile der Datei (alle Annotationen werden dann zu Strings) oder der Import unter `if TYPE_CHECKING:`. Erst ab Python 3.14 ist das verzögerte Auswerten Standard.

In [ ]:
check("l6")

In [ ]:
# hint("l6", 1)

### 🕹 Manueller Check (curses läuft nicht im Notebook)
Im **Terminal** (nicht im PyCharm-Run-Fenster) im Projektordner:
```bash
uv run python -m pysweeper
```
- [ ] Cursor bewegen (Pfeile + WASD), stoppt am Rand
- [ ] Aufdecken, Cascade, Zahlen in Farbe
- [ ] Flagge setzen/entfernen, Minen-Zähler sinkt
- [ ] Timer läuft und stoppt bei Spielende
- [ ] Gewinnen / Verlieren mit Meldung, R startet neu, Q beendet
- [ ] Zu kleines Terminal → Hinweis statt Absturz

### 🤔 Warum?
1. Python kann Duck Typing. Wozu dann überhaupt eine abstrakte Basisklasse?
2. Warum darf `../pysweeper/game.py` nichts von `curses` wissen?
3. Warum wohnt der Cursor in `TerminalUI` und nicht (wie in `main.py`) in `Game`?

In [ ]:
# explain("l6")

---
## Lektion 7 – Bonus: `TkinterUI`, Reflexion & Abschluss

🎯 **Lernziel:** Du beweist dir, dass die Architektur trägt: eine zweite Oberfläche, **ohne** `Game`, `Board` oder `Cell` anzufassen.

### 📖 Die Closure-Falle
🔮 Erst raten, was ausgegeben wird, dann ausführen:

In [ ]:
callbacks = [lambda: print(i) for i in range(3)]
for cb in callbacks:
    cb()

In [ ]:
callbacks = [lambda i=i: print(i) for i in range(3)]   # Wert als Default-Argument einfrieren
for cb in callbacks:
    cb()

Java verbietet diesen Fehler übrigens: Lambdas dürfen nur *effectively final* Variablen benutzen.

### 🛠 Aufgabe 7 (Bonus): `pysweeper/ui/tkinter_ui.py`

| Element | Vertrag |
|---|---|
| `TkinterUI(root)` | erbt von `UserInterface`; `root: tk.Tk`, `buttons: List[List[tk.Button]]` |
| `_create_widgets(width, height)` | ein Button pro Zelle, Linksklick → `_on_left_click(x, y)`, Rechtsklick → `_on_right_click(x, y)` |
| `run(game)` | Widgets bauen, zeichnen, `root.mainloop()` |
| `draw_board(board)` | Button-Texte aus `board.grid` (Farbe über `fg`; macOS-Buttons ignorieren `bg`). Rechtsklick: `"<Button-2>"` (macOS) **und** `"<Button-3>"` (Windows/Linux) binden |

In [ ]:
check("l7")

In [ ]:
# hint("l7", 1)

### 🪞 Reflexion: das Diagramm nachziehen
Dein Code kann inzwischen mehr, als `pysweeper-class-diagram.mmd` zeigt. Ergänze das Diagramm so, dass es **deinen** Code beschreibt: neue Methoden/Properties, private Helfer, das Attribut `game` in `TerminalUI`, …
Danach vergleichen: `explain("l7")`.

### 🧠 Abschluss-Quiz (aus dem Gedächtnis, ohne Zurückblättern)
1. Was erzeugt `@dataclass` automatisch, und was ändert `frozen=True`?
2. Wo entstehen Instanzattribute, und was passiert mit einer Liste, die direkt im Klassenrumpf steht?
3. Was macht `yield` aus einer Funktion?
4. Warum `grid[y][x]` und nicht `grid[x][y]`?
5. Nenne ein Command und eine Query aus deinem `Board`.
6. Was passiert bei `UserInterface()`, und warum ist das gut?
7. Welche Datei führt `python -m pysweeper` aus, und warum braucht sie `if __name__ == "__main__"`?

In [ ]:
# explain("l7"); explain("quiz")

---
## Lektion 8 – `StartMenu`: ein Startmenü als eigene Klasse

🎯 **Lernziel:** Du erbst von einer Klasse aus einer Bibliothek (`tk.Frame`) und weißt, wann und warum `super().__init__(...)` nötig ist. Du bindest Widgets über `tk.StringVar` an Werte und ersetzt ein `if/else` durch ein `dict` mit Funktionen als Werten.

Bisher wählt `__main__.py` Oberfläche und Schwierigkeit in einem curses-Menü aus, und das Menü ist dort als lose Funktionen verstreut. Jetzt bekommt es eine eigene Klasse und ein eigenes Fenster.

### 📖 Von einer Bibliotheksklasse erben
`TkinterUI` **hat** ein Fenster (Komposition: `self.root`). `StartMenu` **ist** ein Frame, also ein Tk-Widget, das man in ein Fenster packen kann (Vererbung). Damit erbt es alles, was ein Frame kann: `pack()`, `winfo_children()`, `destroy()`, …

| Java | Python |
|---|---|
| `class StartPanel extends JPanel` | `class StartMenu(tk.Frame):` |
| `super(layout);` als erste Zeile, sonst setzt der Compiler `super()` selbst ein | `super().__init__(master)`, ruft man es nicht auf, passiert **gar nichts** |
| `super.add(item)` | `super().append(item)` |
| JavaFX `StringProperty` + `bind(...)` | `tk.StringVar` + `variable=...` am Widget |
| `Map<String, Consumer<Difficulty>>` | `dict[str, Callable[[DifficultyConfig], None]]` |

Erst die Liste einrichten, dann das eigene Extra:

In [ ]:
class History(list):
    # Eine Liste, die mitzählt, wie oft etwas angehängt wurde.
    def __init__(self, items=()):
        super().__init__(items)       # erst den geerbten Teil einrichten
        self.appends = 0              # dann das eigene Attribut

    def append(self, item):
        self.appends += 1
        super().append(item)          # geerbte Methode weiter benutzen

h = History(["Kaffee"])
h.append("Tee")
h.append("Wasser")
print(h, len(h), h.appends, isinstance(h, list))

🔮 Erst raten, dann ausführen: Was passiert, wenn die Unterklasse `super().__init__` vergisst?

In [ ]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []

    def add(self, song):
        self.songs.append(song)

class PartyPlaylist(Playlist):
    def __init__(self, name, max_songs):
        self.max_songs = max_songs        # 🐞 super().__init__(name) vergessen

party = PartyPlaylist("Freitag", 50)      # kein Fehler hier ...
try:
    party.add("Song 2")                   # ... sondern erst hier, weit weg von der Ursache
except AttributeError as e:
    print("AttributeError:", e)

Bei `tk.Frame` ist es genauso: Ohne `super().__init__(master)` fehlt dem Objekt sein Tk-Innenleben, und der Fehler kommt erst beim ersten Kind-Widget (`... has no attribute 'tk'`).

### 📖 `tk.StringVar`: ein Wert, an den Widgets gebunden sind
Radiobuttons teilen sich eine Variable. Klickt man einen an, schreibt Tk dessen `value` hinein, und `.get()` liest den aktuellen Wert:

```python
color = tk.StringVar(master=frame, value="rot")          # Startwert
tk.Radiobutton(frame, text="Rot",  value="rot",  variable=color).pack()
tk.Radiobutton(frame, text="Blau", value="blau", variable=color).pack()
...
color.get()                                              # nach Klick auf „Blau“: "blau"
```
(Hier nur als Text: ein Tk-Fenster braucht eine `mainloop()`, die das Notebook blockieren würde.)

### 📖 Funktionen als Werte im `dict`
Eine Funktion ist in Python ein ganz normales Objekt. Ohne `()` dahinter wird sie nicht aufgerufen, sondern weitergereicht:

In [ ]:
def greet_de(name):
    print(f"Hallo {name}!")

def greet_en(name):
    print(f"Hello {name}!")

GREETERS = {"Deutsch": greet_de, "English": greet_en}   # Name → Funktion, ohne ()
print(list(GREETERS))                                    # die Namen, z. B. für ein Menü
GREETERS["English"]("Ada")                               # nachschlagen, dann aufrufen

Eine neue Sprache ist eine neue Zeile im `dict`, kein neuer `elif`-Zweig.

### 🛠 Aufgabe 8: `pysweeper/ui/start_menu.py` und `pysweeper/__main__.py`

| Element | Vertrag |
|---|---|
| `class StartMenu(tk.Frame)` | in `pysweeper/ui/start_menu.py` |
| `StartMenu(master, ui_names, difficulties)` | ruft **zuerst** `super().__init__(master)` auf; speichert `ui_names` und `difficulties`; `ui_choice` und `difficulty_choice` sind `tk.StringVar` mit dem ersten UI-Namen bzw. dem **Namen** der ersten Schwierigkeit als Startwert; `selection = None`; baut die Widgets (`_create_widgets()`) |
| Widgets | ein `tk.Radiobutton` pro UI-Name (Text = Name) an `ui_choice`, einer pro Schwierigkeit (Text enthält `d.name`, `value=d.name`) an `difficulty_choice`, dazu `start_button: tk.Button` → `_on_start`. Verschachteln in `tk.LabelFrame`s ist erlaubt |
| `_on_start()` | `selection = (UI-Name, DifficultyConfig)`, wobei das **Objekt** aus `difficulties` gemeint ist, nicht der Name; danach `self.master.destroy()` |
| `run()` | `self.pack()`, `self.master.mainloop()`, gibt `selection` zurück (`None`, wenn das Fenster ohne Start geschlossen wurde) |
| `__main__.py` | `LAUNCHERS = {"Terminal": ..., "Tkinter": ...}`: Name → Funktion mit dem Parameter `difficulty`. Unter `if __name__ == "__main__":` erst `_use_base_tcl(tk)`, dann `StartMenu(tk.Tk(), list(LAUNCHERS), DIFFICULTIES).run()`, bei einer Auswahl `LAUNCHERS[ui](difficulty)` |
| Aufräumen | das curses-Menü (`choose`, `menu`, `_put`, `UIS`) und die Importe `UP`, `DOWN`, `SELECT` fliegen raus. `run_terminal`, `run_tkinter` und `_use_base_tcl` bleiben |

Die Checks öffnen ein **verstecktes** Tk-Fenster und klicken darin. Vom Menü siehst du dabei nichts.

In [ ]:
check("l8")

In [ ]:
# hint("l8", 1)

### 🕹 Manueller Check
Im **Terminal** im Projektordner (oder über die Run-Konfiguration **pysweeper**):
```bash
uv run python -m pysweeper
```
- [ ] Ein Fenster zeigt Oberfläche und Schwierigkeit zur Auswahl
- [ ] Terminal + Easy → das Fenster schließt sich, das curses-Spiel läuft im Terminal
- [ ] Tkinter + Hard → ein Spielfeld mit 30 × 16 Buttons
- [ ] Fenster ohne Start schließen → das Programm endet ohne Fehlermeldung

### ➕ Erweiterung: zurück ins Startmenü
Nach jedem Spiel soll das Startmenü wieder erscheinen, bis man es ohne Start schließt.

| Element | Vertrag |
|---|---|
| `__main__.py` | Schleife: Menü zeigen, gewählte UI starten, wieder von vorn. Ende, sobald `run()` `None` liefert. Jede Runde braucht ein neues `tk.Tk()`, das alte hat `_on_start` ja geschlossen |

In [ ]:
check("l8_extra")

In [ ]:
# hint("l8_extra", 1)

### 🤔 Warum?
1. `StartMenu` bekommt UI-Namen und Schwierigkeiten übergeben. Warum importiert es sie nicht einfach selbst?
2. `StartMenu` ist eine Oberfläche, erbt aber nicht von `UserInterface`. Warum ist das richtig?
3. Was passiert ohne `super().__init__(master)`, und warum ist der Fehler dann so schwer zu finden?
4. Was gewinnst du mit `LAUNCHERS` gegenüber `if ui == "Terminal": ... else: ...`?

In [ ]:
# explain("l8")

---
## 🏁 Geschafft
Du hast ein 350-Zeilen-Skript in sieben Klassen mit klaren Verantwortlichkeiten zerlegt, jede in ihrer eigenen Datei, getestet und mit zwei austauschbaren Oberflächen, die du über ein eigenes Startmenü auswählst.

**Wie es weitergehen kann:** eigene Tests mit `pytest` schreiben (die Checks in `course/tests/` sind eine gute Vorlage), `mypy` über das Paket laufen lassen, `UserInterface` als `typing.Protocol` umbauen und vergleichen.